In [0]:


####https://cursos.alura.com.br/course/testes-estatisticos-comparacao/task/20934

Este projeto aplica métodos de estatística avançada para comparar o engajamento de usuários em diferentes trailers de filmes em uma plataforma de streaming. 

A análise utiliza amostras de usuários e o tempo médio de visualização como métrica de engajamento. Inicialmente, é realizada a comparação entre dois grupos usando o teste t para médias. Em seguida, com a inclusão de um terceiro trailer, aplica-se ANOVA para comparar múltiplos grupos.

O objetivo é demonstrar como técnicas de inferência estatística podem ser usadas para apoiar decisões baseadas em dados.



1. **ANOVA One-Way:** Descobrir se há alguma diferença estatisticamente significativa entre as médias de visualização dos três trailers (Drama, Comédia e Futurista).
2. **Teste de Tukey HSD:** Se a ANOVA nos der um sinal verde, usaremos este teste para descobrir **onde** as diferenças se encontram.
3. **Teste de Bonferroni:** Exploraremos uma alternativa ao Tukey, entendendo suas vantagens e desvantagens


In [0]:
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.stats.anova as anova
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import f

In [0]:
df = pd.read_csv("/Workspace/Repos/patriciatamiresdesousa@gmail.com/estatistica-avancada-comparacao-entre-grupos/trailers_dados_aulas_1_e_2.csv")
df.head()

In [0]:

## Visualizacao dos dados dos trailers

trailer_types = df['trailer_tipo'].unique()
fig, axes = plt.subplots(1, len(trailer_types), figsize=(15, 5))
for i, trailer_type in enumerate(trailer_types):
    subset_df = df[df['trailer_tipo'] == trailer_type]
    sns.histplot(subset_df['tempo_visualizacao'], ax=axes[i], kde=True)
    axes[i].set_title(f'Distribuição de Tempo de Visualização - {trailer_type}')
    axes[i].set_xlabel('Tempo de Visualização')
    axes[i].set_ylabel('Frequência')
y_max = max([ax.get_ylim()[1] for ax in axes])
for ax in axes:
    ax.set_ylim(0, y_max)
    
x_min = df['tempo_visualizacao'].min()
x_max = df['tempo_visualizacao'].max()
for ax in axes:
    ax.set_xlim(x_min, x_max)

plt.tight_layout()
plt.show()

In [0]:
#### Tabela estatistica
### tanto a media como o desvio padrao esta altas para o trailer futurista 


tabela_estat = df.groupby('trailer_tipo')['tempo_visualizacao'].agg(['mean', 'std', 'count']).reset_index()
tabela_estat.rename(columns={'mean': 'media', 'std': 'desvio_padrao', 'count': 'numero_observacoes'}, inplace=True)
display(tabela_estat)

In [0]:
##Tabela ANOVA


##Resulto F grande o que siginifica que a diferença entre os trailer
## Aqui como avalio a diferença, o valor F avalia a diferenca eentre os grupos sendo essa diferença parecida ou não
## quando as médias são proximas o valor F é baixo, quando as médias são altas os valores são altos o que siginifca uma diferença entre os trailers
##Outro ponto importante o P-Value(>P) quando menor maior a diferenca siginfiicativa dos grupos, quando maior, maior a diferença pode ser aleatoria. Aqui a diferença é de 1.63e-19 = 1,63x10-19
###0.000000000000000000163 muito menor que 0.05


modelo = smf.ols('tempo_visualizacao ~ C(trailer_tipo)', data=df).fit()
anova_tabela = anova.anova_lm(modelo, typ=2)

print(anova_table)

In [0]:

###Visualizando a distribuição F

df_dados_anova = anova_table.loc['C(trailer_tipo)', 'df']
df_resido = anova_table.loc['Residual', 'df']


x = np.linspace(f.ppf(0.001, df_dados_anova, df_resido),
                f.ppf(0.999, df_dados_anova, df_resido), 100)
                

plt.figure(figsize=(10, 6))
plt.plot(x, f.pdf(x, df_dados_anova, df_resido),
         'r-', lw=2, alpha=0.6, label=f'df1={int(df_dados_anova)}, df2={int(df_resido)}')

f_calculated = anova_table.loc['C(trailer_tipo)','F']
plt.axvline(f_calculated, color='blue', linestyle='dashed', linewidth=2, label=f'F Calculado = {f_calculated:.2f}')


plt.title('Distribuição F com Graus de Liberdade do Teste ANOVA')
plt.xlabel('Valor de F')
plt.ylabel('Densidade de Probabilidade')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

####Teste de Tukey 

##### Um teste estatístico usado depois de uma ANOVA para descobrir quais grupos são diferentes entre si.

In [0]:
#### Teste Tukey= Objetivo é encontrar qual grupo tem diferença 
### pairwise_tukeyhsd aplica o teste enqaunto o parametro alpha é definifo para representar a significação do teste 

teste_tukey = pairwise_tukeyhsd(endog=df['tempo_visualizacao'], groups=df['trailer_tipo'], alpha=0.05)
print(teste_tukey.summary())